# 02 — Query 1 (Grouping & Fatality Rates) & Query 2 (Frequent Pattern Mining / Apriori)

- **Owner:** Baptiste Noailhac (`@BaptisteNoai`)
- **Course Rubric:** Query 1 `[1 pt]` + Query 2 `[2 pts]` (reusing Lab 9 `mlxtend` `apriori` and `association_rules`)

In [1]:
# Import libraries and project functions
import os
import sys
import warnings
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_shark_data, clean_shark_data
from src.indicators import query_grouping_analysis, query_frequent_patterns

warnings.filterwarnings('ignore')
clean_sharks = clean_shark_data(load_shark_data('../data/attacks.csv'))
print(f'Loaded {len(clean_sharks)} cleaned shark attack incidents (1960-2018).')

Loaded 4033 cleaned shark attack incidents (1960-2018).


## Query 1 — Grouping & Fatality Rate Analysis `[1 pt]`

### Indicator Explanation
- **Calculation:** Group cleaned incidents by `Country_clean`, `Activity_group`, and `Species_group`, computing `Total_Attacks = count(Case Number)`, `Fatal_Attacks = sum(Fatal_clean == 'Y')`, and:
  $$\text{Fatality Rate (\%)} = \frac{\text{Fatal Attacks}}{\text{Total Attacks}} \times 100$$
- **What it represents:** Measures both the absolute frequency of shark encounters and their lethality across nations, water activities, and shark species.
- **Interpretation:**
  - **Countries:** The United States dominates raw volume (1,756 attacks) but has a remarkably low fatality rate (~4.5%), whereas Australia (704), South Africa (432), and Brazil (103) show substantially higher lethality.
  - **Activities:** Surfing generates the most encounters (1,250) yet exhibits low mortality because boards act as physical shields and bites are typically exploratory, whereas Swimming and Diving/Snorkeling face much higher fatality rates.
  - **Species:** Among identified incidents, **Tiger Shark (22.3%)**, **White Shark (18.0%)**, and **Bull Shark (16.0%)** account for almost all lethal encounters, while **Reef/Nurse** and **Blacktip/Spinner** sharks show 0% fatality.

In [2]:
# Execute Query 1 grouping indicators and display summary tables + Plotly charts
q1 = query_grouping_analysis(clean_sharks, top_n=10)

print('--- Top 10 Countries ---')
display(q1['top_countries'])
q1['fig_countries'].show()

print('--- Activity Group Fatality Rates ---')
display(q1['activity_stats'])
q1['fig_activities'].show()

print('--- Shark Species Fatality Rates ---')
display(q1['species_stats'])
q1['fig_species'].show()

--- Top 10 Countries ---


,Country_clean,Total_Attacks,Fatal_Attacks,Non_Fatal_Attacks,Fatality_Rate_Pct
0,United States,1756,63,1544,3.59
1,Australia,704,76,561,10.80
2,South Africa,432,48,336,11.11
3,Brazil,103,31,63,30.10
4,"Bahamas, The",91,4,82,4.40
5,New Zealand,79,9,63,11.39
6,Papua New Guinea,61,24,34,39.34
7,Reunion,56,26,30,46.43
8,Mexico,54,20,27,37.04
9,New Caledonia,46,14,31,30.43


--- Activity Group Fatality Rates ---


,Activity_group,Total_Attacks,Fatal_Attacks,Fatality_Rate_Pct
0,Surfing,1257,66,5.25
1,Diving/Snorkeling,775,122,15.74
2,Other,629,109,17.33
3,Swimming,623,138,22.15
4,Fishing,400,37,9.25
5,Wading,241,12,4.98
6,Boating/Kayaking,108,11,10.19


--- Shark Species Fatality Rates ---


,Species_group,Total_Attacks,Fatal_Attacks,Fatality_Rate_Pct
0,Unidentified,1730,264,15.26
1,Other Named Shark,850,49,5.76
2,White Shark,561,101,18.00
3,Tiger Shark,220,49,22.27
4,Hammerhead/Mako/Whaler,208,7,3.37
5,Reef/Nurse Shark,190,0,0.00
6,Bull Shark,156,25,16.03
7,Blacktip/Spinner Shark,118,0,0.00


## Query 2 — Frequent Pattern Mining with Apriori `[2 pts]`

### Indicator Explanation
- **Calculation:** Encoded 6 discrete attributes (`Country_clean` top-5 + Other, `Activity_group`, `Type`, `Season`, `Time_of_day`, `Fatal_clean`) into a boolean one-hot transaction matrix after dropping `Unknown` entries. Swept `min_support` from `0.15` down to `0.02` (Lab 9 sensitivity curve), extracted frequent itemsets at `min_support = 0.03`, and generated association rules filtered by `lift >= 1.05`:
  $$\text{Support}(X) = \frac{\text{count}(X)}{N}, \quad \text{Confidence}(X \rightarrow Y) = \frac{\text{Support}(X \cup Y)}{\text{Support}(X)}, \quad \text{Lift}(X \rightarrow Y) = \frac{\text{Confidence}(X \rightarrow Y)}{\text{Support}(Y)}$$
- **What it represents:** Uncovers recurring multi-factor combinations of geography, season, time of day, activity, and outcome.
- **Interpretation:** The sensitivity curve reveals the classic exponential growth of candidate itemsets below `min_support = 0.05`. High-lift rules highlight distinct regional profiles — for instance, US summer afternoon surfing incidents are strongly associated with non-fatal unprovoked encounters, whereas diving/spearfishing rules carry higher associations with non-US coastal zones.

In [3]:
# Execute Query 2 Apriori frequent pattern mining and inspect top rules
q2 = query_frequent_patterns(clean_sharks, min_support=0.03, min_lift=1.05)

print(f"Complete transactions analyzed: {q2['n_transactions']}")
print(f"Frequent itemsets at min_support=0.03: {len(q2['frequent_itemsets'])}")
print(f"Association rules at min_lift=1.05: {len(q2['rules'])}")

display(q2['sensitivity_df'])
q2['fig_sensitivity'].show()

print('--- Top 15 Association Rules Sorted by Lift ---')
display(q2['rules'][['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head(15))
q2['fig_rules'].show()

Complete transactions analyzed: 2277
Frequent itemsets at min_support=0.03: 417
Association rules at min_lift=1.05: 2262


,min_support,pattern_count
0,0.15,63
1,0.10,118
2,0.08,157
3,0.05,263
4,0.04,328
5,0.03,417
6,0.02,580


--- Top 15 Association Rules Sorted by Lift ---


,antecedents_str,consequents_str,support,confidence,lift
0,Fatal_clean_Y,"Country_clean_Other_Country, Type_Unprovoked",0.045674,0.400000,2.957143
1,"Country_clean_Other_Country, Type_Unprovoked",Fatal_clean_Y,0.045674,0.337662,2.957143
2,Fatal_clean_Y,Country_clean_Other_Country,0.048748,0.426923,2.730629
3,Country_clean_Other_Country,Fatal_clean_Y,0.048748,0.311798,2.730629
4,"Fatal_clean_Y, Type_Unprovoked",Country_clean_Other_Country,0.045674,0.426230,2.726193
5,Country_clean_Other_Country,"Fatal_clean_Y, Type_Unprovoked",0.045674,0.292135,2.726193
6,"Country_clean_United States, Season_Summer, Ty...","Activity_group_Wading, Fatal_clean_N",0.032499,0.173302,2.242096
7,"Activity_group_Wading, Fatal_clean_N","Country_clean_United States, Season_Summer, Ty...",0.032499,0.420455,2.242096
8,Activity_group_Wading,"Country_clean_United States, Fatal_clean_N, Se...",0.032499,0.395722,2.155643
9,"Country_clean_United States, Fatal_clean_N, Se...",Activity_group_Wading,0.032499,0.177033,2.155643


In [4]:
# Focused view: strongest rules predicting Fatality status (Fatal_clean_N or Fatal_clean_Y)
rules_df = q2['rules']
fatal_rules = rules_df[rules_df['consequents_str'].isin(['Fatal_clean_N', 'Fatal_clean_Y'])].head(10)
print('Top rules predicting Fatal_clean outcome:')
display(fatal_rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']])

Top rules predicting Fatal_clean outcome:


,antecedents_str,consequents_str,support,confidence,lift
1,"Country_clean_Other_Country, Type_Unprovoked",Fatal_clean_Y,0.045674,0.337662,2.957143
3,Country_clean_Other_Country,Fatal_clean_Y,0.048748,0.311798,2.730629
41,"Activity_group_Diving/Snorkeling, Type_Unprovoked",Fatal_clean_Y,0.031621,0.211144,1.849132
76,Activity_group_Swimming,Fatal_clean_Y,0.032938,0.199468,1.746880
103,"Activity_group_Swimming, Type_Unprovoked",Fatal_clean_Y,0.031181,0.196133,1.717669
125,Activity_group_Diving/Snorkeling,Fatal_clean_Y,0.032060,0.192105,1.682399
1386,"Time_of_day_Morning (06-12), Type_Unprovoked",Fatal_clean_Y,0.038208,0.130827,1.145743
1466,"Activity_group_Wading, Country_clean_United St...",Fatal_clean_N,0.058410,1.000000,1.128904
1467,"Time_of_day_Afternoon (12-18), Type_Provoked",Fatal_clean_N,0.035134,1.000000,1.128904
1468,"Activity_group_Wading, Country_clean_United St...",Fatal_clean_N,0.057093,1.000000,1.128904
